In [1]:
import csv

with open("online_retail.xlsx", "r", encoding="ISO-8859-1") as f:
    sample = f.read(1000)
    print(csv.Sniffer().sniff(sample).delimiter)

,


In [2]:
import pandas as pd

df = pd.read_excel("online_retail.xlsx")

print(df.head())
print(df.shape)

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
(541909, 8)


In [3]:
import pandas as pd

# Remove rows with no customer
df = df.dropna(subset=["CustomerID"])

# Remove returns (negative quantity)
df = df[df["Quantity"] > 0]

# Remove cancelled invoices
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]

# Create TotalPrice
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

# Convert date properly (already mostly correct, but safe)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print("Cleaned shape:", df.shape)

Cleaned shape: (397924, 9)


In [4]:
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month

In [5]:
import pyodbc

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=ecom-ml.database.windows.net;"
    "DATABASE=free-sql-db-4249307;"
    "UID=CloudSA90e78dbc;"
    "PWD=haru@2003;"
    "Encrypt=yes;TrustServerCertificate=no;"
)

cursor = conn.cursor()
print("Connected")

Connected


In [6]:
cursor.execute("SELECT DB_NAME()")
print(cursor.fetchone())

('free-sql-db-4249307',)


In [8]:
cursor.executemany(
    "INSERT INTO dbo.test_table (name, age) VALUES (?, ?)",
    [
        ("Sam", 22),
        ("Alex", 25),
        ("John", 30)
    ]
)
conn.commit()
print("Inserted")

Inserted


In [9]:
cursor.execute("SELECT * FROM dbo.test_table")
print(cursor.fetchall())

[('Sam', 22), ('Alex', 25), ('John', 30), ('Sam', 22), ('Alex', 25), ('John', 30)]


In [10]:
print(df.columns)
print(len(df.columns))



Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'TotalPrice', 'Year', 'Month'],
      dtype='object')
11


In [11]:
df_upload = df[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country",
    ]
]

df_upload.to_csv("online_retail_clean.csv", index=False)
print("CSV saved")

CSV saved


In [12]:
cursor.execute("""
IF OBJECT_ID('dbo.online_retail_clean', 'U') IS NOT NULL
DROP TABLE dbo.online_retail_clean

CREATE TABLE dbo.online_retail_clean (
    InvoiceNo NVARCHAR(20),
    StockCode NVARCHAR(20),
    Description NVARCHAR(255),
    Quantity INT,
    InvoiceDate DATETIME,
    UnitPrice FLOAT,
    CustomerID FLOAT,
    Country NVARCHAR(50)
)
""")

conn.commit()

print("Table ready")

Table ready


In [13]:
import pyodbc
import pandas as pd

df = pd.read_csv("online_retail_clean.csv")

cursor.fast_executemany = True

cursor.executemany("""
INSERT INTO dbo.online_retail_clean
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", df.values.tolist())

conn.commit()

print("Bulk upload complete 🚀")

Bulk upload complete 🚀


In [14]:

cursor.execute("SELECT COUNT(*) FROM dbo.online_retail_clean")
print(cursor.fetchone())


(397924,)


In [15]:
cursor.execute("""
SELECT TOP 10 *
FROM dbo.online_retail_clean
""")

for row in cursor.fetchall():
    print(row)

('536365', '85123A', 'WHITE HANGING HEART T-LIGHT HOLDER', 6, datetime.datetime(2010, 12, 1, 8, 26), 2.55, 17850.0, 'United Kingdom')
('536365', '71053', 'WHITE METAL LANTERN', 6, datetime.datetime(2010, 12, 1, 8, 26), 3.39, 17850.0, 'United Kingdom')
('536365', '84406B', 'CREAM CUPID HEARTS COAT HANGER', 8, datetime.datetime(2010, 12, 1, 8, 26), 2.75, 17850.0, 'United Kingdom')
('536365', '84029G', 'KNITTED UNION FLAG HOT WATER BOTTLE', 6, datetime.datetime(2010, 12, 1, 8, 26), 3.39, 17850.0, 'United Kingdom')
('536365', '84029E', 'RED WOOLLY HOTTIE WHITE HEART.', 6, datetime.datetime(2010, 12, 1, 8, 26), 3.39, 17850.0, 'United Kingdom')
('536365', '22752', 'SET 7 BABUSHKA NESTING BOXES', 2, datetime.datetime(2010, 12, 1, 8, 26), 7.65, 17850.0, 'United Kingdom')
('536365', '21730', 'GLASS STAR FROSTED T-LIGHT HOLDER', 6, datetime.datetime(2010, 12, 1, 8, 26), 4.25, 17850.0, 'United Kingdom')
('536366', '22633', 'HAND WARMER UNION JACK', 6, datetime.datetime(2010, 12, 1, 8, 28), 1.85, 

In [16]:
cursor.execute("""
SELECT
    InvoiceNo,
    Quantity,
    UnitPrice,
    Quantity * UnitPrice AS Revenue
FROM dbo.online_retail_clean
""")

print(cursor.fetchmany(5))

[('536365', 6, 2.55, 15.299999999999999), ('536365', 6, 3.39, 20.34), ('536365', 8, 2.75, 22.0), ('536365', 6, 3.39, 20.34), ('536365', 6, 3.39, 20.34)]


In [17]:
cursor.execute("""
SELECT
    Country,
    SUM(Quantity * UnitPrice) AS TotalRevenue
FROM dbo.online_retail_clean
GROUP BY Country
ORDER BY TotalRevenue DESC
""")

for row in cursor.fetchall():
    print(row)

('United Kingdom', 7308391.5540030645)
('Netherlands', 285446.34)
('EIRE', 265545.8999999985)
('Germany', 228867.13999999844)
('France', 209024.0499999997)
('Australia', 138521.30999999956)
('Spain', 61577.10999999997)
('Switzerland', 56443.95000000004)
('Belgium', 41196.33999999997)
('Sweden', 38378.33000000003)
('Japan', 37416.37000000001)
('Norway', 36165.440000000046)
('Portugal', 33439.890000000014)
('Finland', 22546.079999999973)
('Singapore', 21279.289999999994)
('Channel Islands', 20450.439999999962)
('Denmark', 18955.339999999997)
('Italy', 17483.240000000005)
('Cyprus', 13590.379999999994)
('Austria', 10198.679999999997)
('Poland', 7334.649999999997)
('Israel', 7221.689999999999)
('Greece', 4760.5199999999995)
('Iceland', 4309.999999999997)
('Canada', 3666.380000000001)
('USA', 3580.3900000000012)
('Malta', 2725.59)
('Unspecified', 2667.0700000000006)
('United Arab Emirates', 1902.2800000000007)
('Lebanon', 1693.8800000000003)
('Lithuania', 1661.06)
('European Community', 130

In [18]:
cursor.execute("""
IF OBJECT_ID('dbo.monthly_revenue', 'U') IS NOT NULL
DROP TABLE dbo.monthly_revenue

SELECT
    YEAR(InvoiceDate) AS Year,
    MONTH(InvoiceDate) AS Month,
    SUM(Quantity * UnitPrice) AS Revenue,
    COUNT(DISTINCT CustomerID) AS Customers,
    COUNT(DISTINCT InvoiceNo) AS Orders
INTO dbo.monthly_revenue
FROM dbo.online_retail_clean
GROUP BY YEAR(InvoiceDate), MONTH(InvoiceDate)
""")

conn.commit()
print("Monthly table created")

Monthly table created


In [19]:
cursor.execute("SELECT * FROM dbo.monthly_revenue ORDER BY Year, Month")

for row in cursor.fetchall():
    print(row)

(2010, 12, 572713.8900000163, 885, 1400)
(2011, 1, 569445.0400000077, 741, 987)
(2011, 2, 447137.3500000165, 758, 998)
(2011, 3, 595500.760000013, 974, 1321)
(2011, 4, 469200.3610000132, 856, 1149)
(2011, 5, 678594.5600000018, 1056, 1555)
(2011, 6, 661213.6900000116, 991, 1393)
(2011, 7, 600091.0110000141, 949, 1331)
(2011, 8, 645343.900000009, 935, 1281)
(2011, 9, 952838.3819999964, 1266, 1756)
(2011, 10, 1039318.7899999822, 1364, 1929)
(2011, 11, 1161817.3799999433, 1665, 2658)
(2011, 12, 518192.7900000037, 615, 778)
